In [1]:
import sys
import os
import pandas as pd

In [2]:
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

In [4]:
from src.utils.config_loader import load_config
from src.agent.nodes.critic import critic_node

cfg = load_config()

real_data_path = os.path.join(project_root, cfg.path.data.test)
df = pd.read_csv(real_data_path)
row = df.iloc[501]

In [5]:
print(row) # Dummy data 전체 형태

Unnamed: 0                                                     501
id                                          generation-for-nlp-711
paragraph        현재 가처분 소득이 10,000달러이고 소비 지출이 8,000달러라고 가정하겠습니다...
problems         {'question': '이 정보가 주어지면?', 'choices': ['한계 소비...
question_plus                                                  NaN
Name: 501, dtype: object


In [6]:
import ast

paragraph = row.get('paragraph', '') # 지문

problem = row.get("problems", {})
if isinstance(problem, str):
    problem_dict = ast.literal_eval(problem)
else:
    problem_dict = problem

question = problem_dict.get('question', '') # 질문
choices = problem_dict.get('choices', []) # 선지

print(f"🔹 지문: {paragraph}")
print(f"🔹 질문: {question}")
print(f"🔹 선택지: {choices}")


🔹 지문: 현재 가처분 소득이 10,000달러이고 소비 지출이 8,000달러라고 가정하겠습니다. 가처분 소득이 100달러 증가할 때마다 저축액은 10달러 증가합니다.
🔹 질문: 이 정보가 주어지면?
🔹 선택지: ['한계 소비 성향은 0.80이다.', '한계 저축 성향은 0.20이다.', '한계 저축 성향은 0.10이다.', '한계 저축 성향은 0.90이다.']


In [7]:
dummy_state = {
    "paragraph": paragraph,
    "problem": {
        "question": question,
        "choices": choices
    },
    "solver_results": [
        {
            # 케이스 1: 정답(3번: 0.10)은 맞지만 근거가 빈약함 (논리 부족)
            "answer": "3",
            "reasoning": "지문에서 소득과 저축의 변화가 언급되어 있으며, 보기 중 0.10이 가장 수치적으로 타당해 보입니다." 
        },
        {
            # 케이스 2: 정답이 틀렸으며(1번) 근거도 오류인 경우 (계산 실수/논리 오류)
            "answer": "1",
            "reasoning": "현재 가처분 소득 10,000달러 중 소비 지출이 8,000달러이므로 비율은 0.8입니다. 따라서 소득 증가 시 소비 성향도 0.80이 됩니다." 
        },
        {
            # 케이스 3: 정답(3번)과 근거(MPS 계산)가 모두 완벽함 (Pass 기대)
            "answer": "3",
            "reasoning": "한계 저축 성향(MPS)은 소득 변화량에 대한 저축 변화량의 비율입니다. 지문에 따르면 소득 100달러 증가 시 저축이 10달러 증가하므로, MPS = 10 / 100 = 0.10입니다. 따라서 '한계 저축 성향은 0.10이다'라는 3번 선지가 정답입니다." 
        }
    ]
}

In [8]:
# 노드 초기화
print("⏳ CriticNode 초기화 중... (모델 로딩)")
CriticNode = critic_node(cfg)
# 노드 실행
print(f"▶️ 검증 시작 (총 {len(dummy_state['solver_results'])}개의 결과)")
result = CriticNode(dummy_state)

⏳ CriticNode 초기화 중... (모델 로딩)
🔧 [critic_node] 초기화 중... (사용 모델: main_solver)
🔄 [Loader] 모델 로딩 시작: Qwen3-32B-4bit (unsloth/Qwen3-32B-bnb-4bit)
   ↳ ⚡ 양자화 설정 적용 중...


config.json: 0.00B [00:00, ?B/s]

/data/ephemeral/home/dh_dir/pro-nlp-generationfornlp-nlp-08/.venv/lib/python3.11/site-packages/transformers/quantizers/auto.py:239: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

✅ [Loader] 로딩 완료!
▶️ 검증 시작 (총 3개의 결과)
{"critic_reason": "지문에 따르면 가처분 소득이 100달러 증가할 때 저축액이 10달러 증가하므로 한계 저축 성향은 10/100 = 0.10이다. 따라서 정답 선정 이유는 수치적 계산에 기반하여 논리적이다.", "critic_result": "Pass"}<|im_end|>
```json
{"critic_reason": "모델은 한계 소비 성향을 계산할 때 초기 소득 대비 소비 비율을 사용했으나, 한계 소비 성향은 소득 증가에 따른 소비 증가 비율을 의미함. 지문에 따르면 100달러 증가 시 10달러가 저축되므로 90달러가 소비되므로 한계 소비 성향은 0.90임.", "critic_result": "Fail"}
```<|im_end|>
```json
{"critic_reason": "지문에 따르면 소득이 100달러 증가할 때 저축이 10달러 증가하므로, 한계 저축 성향(MPS)은 10/100 = 0.10이다. 이는 3번 선지와 일치하며, 논리적으로 타당하다.", "critic_result": "Pass"}
```<|im_end|>


In [12]:
print(result)

{'critic_results': [{'critic_reason': '지문에 따르면 가처분 소득이 100달러 증가할 때 저축액이 10달러 증가하므로 한계 저축 성향은 10/100 = 0.10이다. 따라서 정답 선정 이유는 수치적 계산에 기반하여 논리적이다.', 'critic_result': 'Pass'}, {'critic_reason': '모델은 한계 소비 성향을 계산할 때 초기 소득 대비 소비 비율을 사용했으나, 한계 소비 성향은 소득 증가에 따른 소비 증가 비율을 의미함. 지문에 따르면 100달러 증가 시 10달러가 저축되므로 90달러가 소비되므로 한계 소비 성향은 0.90임.', 'critic_result': 'Fail'}, {'critic_reason': '지문에 따르면 소득이 100달러 증가할 때 저축이 10달러 증가하므로, 한계 저축 성향(MPS)은 10/100 = 0.10이다. 이는 3번 선지와 일치하며, 논리적으로 타당하다.', 'critic_result': 'Pass'}]}


In [13]:
# 결과 출력
print("\n" + "="*50)
print("🧐 CriticNode 검증 결과 요약")
print("="*50)

for i, report in enumerate(result['critic_results']):
    orig_res = dummy_state['solver_results'][i]
    print(f"\n[결과 {i+1}]")
    print(f"🔹 모델 답변: {orig_res['answer']}")
    print(f"🔹 선정 이유: {orig_res['reasoning']}")
    print(f"--------------------------------------------------")
    print(f"🚩 판정: {report['critic_result']}")
    print(f"📝 근거: {report['critic_reason']}")
    print("-" * 50)


🧐 CriticNode 검증 결과 요약

[결과 1]
🔹 모델 답변: 3
🔹 선정 이유: 지문에서 소득과 저축의 변화가 언급되어 있으며, 보기 중 0.10이 가장 수치적으로 타당해 보입니다.
--------------------------------------------------
🚩 판정: Pass
📝 근거: 지문에 따르면 가처분 소득이 100달러 증가할 때 저축액이 10달러 증가하므로 한계 저축 성향은 10/100 = 0.10이다. 따라서 정답 선정 이유는 수치적 계산에 기반하여 논리적이다.
--------------------------------------------------

[결과 2]
🔹 모델 답변: 1
🔹 선정 이유: 현재 가처분 소득 10,000달러 중 소비 지출이 8,000달러이므로 비율은 0.8입니다. 따라서 소득 증가 시 소비 성향도 0.80이 됩니다.
--------------------------------------------------
🚩 판정: Fail
📝 근거: 모델은 한계 소비 성향을 계산할 때 초기 소득 대비 소비 비율을 사용했으나, 한계 소비 성향은 소득 증가에 따른 소비 증가 비율을 의미함. 지문에 따르면 100달러 증가 시 10달러가 저축되므로 90달러가 소비되므로 한계 소비 성향은 0.90임.
--------------------------------------------------

[결과 3]
🔹 모델 답변: 3
🔹 선정 이유: 한계 저축 성향(MPS)은 소득 변화량에 대한 저축 변화량의 비율입니다. 지문에 따르면 소득 100달러 증가 시 저축이 10달러 증가하므로, MPS = 10 / 100 = 0.10입니다. 따라서 '한계 저축 성향은 0.10이다'라는 3번 선지가 정답입니다.
--------------------------------------------------
🚩 판정: Pass
📝 근거: 지문에 따르면 소득이 100달러 증가할 때 저축이 10달러 증가하므로, 한계 저축 성향(MPS)은 10/100 